# Lesson 2: RAG Triad of metrics

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import utils

import os
import openai
openai.api_key = utils.get_openai_api_key()

In [3]:
from trulens_eval import Tru

tru = Tru()
tru.reset_database()

🦑 Tru initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `Tru` to prevent this.


In [4]:
from llama_index import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()

In [5]:
from llama_index import Document

document = Document(text="\n\n".\
                    join([doc.text for doc in documents]))

In [6]:
from utils import build_sentence_window_index

from llama_index.llms import OpenAI

llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)

sentence_index = build_sentence_window_index(
    document,
    llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="sentence_index"
)

In [7]:
from utils import get_sentence_window_query_engine

sentence_window_engine = \
get_sentence_window_query_engine(sentence_index)

In [8]:
output = sentence_window_engine.query(
    "How do you create your AI portfolio?")
output.response

'You create your AI portfolio by building a collection of projects that demonstrate a progression of skills in the field.'

## Feedback functions

In [9]:
import nest_asyncio

nest_asyncio.apply()

In [10]:
from trulens_eval import OpenAI as fOpenAI

provider = fOpenAI()

### 1. Answer Relevance

In [11]:
from trulens_eval import Feedback

f_qa_relevance = Feedback(
    provider.relevance_with_cot_reasons,
    name="Answer Relevance"
).on_input_output()

✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .


### 2. Context Relevance

In [12]:
from trulens_eval import TruLlama

context_selection = TruLlama.select_source_nodes().node.text

In [13]:
import numpy as np

f_qs_relevance = (
    Feedback(provider.qs_relevance,
             name="Context Relevance")
    .on_input()
    .on(context_selection)
    .aggregate(np.mean)
)

✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input statement will be set to __record__.app.query.rets.source_nodes[:].node.text .


In [14]:
import numpy as np

f_qs_relevance = (
    Feedback(provider.qs_relevance_with_cot_reasons,
             name="Context Relevance")
    .on_input()
    .on(context_selection)
    .aggregate(np.mean)
)

✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input statement will be set to __record__.app.query.rets.source_nodes[:].node.text .


### 3. Groundedness

In [15]:
from trulens_eval.feedback import Groundedness

grounded = Groundedness(groundedness_provider=provider)

In [16]:
f_groundedness = (
    Feedback(grounded.groundedness_measure_with_cot_reasons,
             name="Groundedness"
            )
    .on(context_selection)
    .on_output()
    .aggregate(grounded.grounded_statements_aggregator)
)

✅ In Groundedness, input source will be set to __record__.app.query.rets.source_nodes[:].node.text .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .


## Evaluation of the RAG application

In [17]:
from trulens_eval import TruLlama
from trulens_eval import FeedbackMode

tru_recorder = TruLlama(
    sentence_window_engine,
    app_id="App_1",
    feedbacks=[
        f_qa_relevance,
        f_qs_relevance,
        f_groundedness
    ]
)

In [18]:
eval_questions = []
with open('eval_questions.txt', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        eval_questions.append(item)

In [19]:
eval_questions

['What are the keys to building a career in AI?',
 'How can teamwork contribute to success in AI?',
 'What is the importance of networking in AI?',
 'What are some good habits to develop for a successful career?',
 'How can altruism be beneficial in building a career?',
 'What is imposter syndrome and how does it relate to AI?',
 'Who are some accomplished individuals who have experienced imposter syndrome?',
 'What is the first step to becoming good at AI?',
 'What are some common challenges in AI?',
 'Is it normal to find parts of AI challenging?']

In [20]:
eval_questions.append("How can I be successful in AI?")

In [21]:
eval_questions

['What are the keys to building a career in AI?',
 'How can teamwork contribute to success in AI?',
 'What is the importance of networking in AI?',
 'What are some good habits to develop for a successful career?',
 'How can altruism be beneficial in building a career?',
 'What is imposter syndrome and how does it relate to AI?',
 'Who are some accomplished individuals who have experienced imposter syndrome?',
 'What is the first step to becoming good at AI?',
 'What are some common challenges in AI?',
 'Is it normal to find parts of AI challenging?',
 'How can I be successful in AI?']

In [22]:
for question in eval_questions:
    with tru_recorder as recording:
        sentence_window_engine.query(question)

In [23]:
records, feedback = tru.get_records_and_feedback(app_ids=[])
records.head()

,app_id,app_json,type,record_id,input,output,tags,record_json,cost_json,perf_json,ts,Answer Relevance,Context Relevance,Groundedness,Answer Relevance_calls,Context Relevance_calls,Groundedness_calls,latency,total_tokens,total_cost
0,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_3990d590b1d1351d5f62365c80033129,"""What are the keys to building a career in AI?""","""The keys to building a career in AI involve l...",-,"{""record_id"": ""record_hash_3990d590b1d1351d5f6...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-03-31T09:34:35.019585"", ""...",2026-03-31T09:34:36.951717,1.0,0.9,0.5,[{'args': {'prompt': 'What are the keys to bui...,[{'args': {'question': 'What are the keys to b...,[{'args': {'source': 'Chapter 7: A Simple Fram...,1,509,0.000779
1,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_9d9074dcec43af1d37a614acae82edd2,"""How can teamwork contribute to success in AI?""","""Teammates play a crucial role in the success ...",-,"{""record_id"": ""record_hash_9d9074dcec43af1d37a...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-03-31T09:34:37.099838"", ""...",2026-03-31T09:34:40.586038,0.9,0.8,0.6,[{'args': {'prompt': 'How can teamwork contrib...,[{'args': {'question': 'How can teamwork contr...,[{'args': {'source': 'To get a project starte...,3,635,0.000992
2,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_950d35e9848ac573f280b5724098449a,"""What is the importance of networking in AI?""","""Networking in AI is crucial as it can provide...",-,"{""record_id"": ""record_hash_950d35e9848ac573f28...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-03-31T09:34:40.713638"", ""...",2026-03-31T09:34:42.996311,1.0,0.8,0.0,[{'args': {'prompt': 'What is the importance o...,[{'args': {'question': 'What is the importance...,[{'args': {'source': 'What is the hiring proce...,2,533,0.000849
3,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_be06b1a8168d739f5897c01c8a1cce5c,"""What are some good habits to develop for a su...","""Developing good habits in areas such as eatin...",-,"{""record_id"": ""record_hash_be06b1a8168d739f589...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-03-31T09:34:43.120377"", ""...",2026-03-31T09:34:44.812930,1.0,NaN,NaN,[{'args': {'prompt': 'What are some good habit...,NaN,NaN,1,434,0.000669
4,App_1,"{""app_id"": ""App_1"", ""tags"": ""-"", ""metadata"": {...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_51477ce0e11bc87395611b1459e3a956,"""How can altruism be beneficial in building a ...","""Helping others and lifting them up during one...",-,"{""record_id"": ""record_hash_51477ce0e11bc873956...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-03-31T09:34:44.939336"", ""...",2026-03-31T09:34:46.360478,1.0,NaN,NaN,[{'args': {'prompt': 'How can altruism be bene...,NaN,NaN,1,482,0.000734


In [24]:
import pandas as pd

pd.set_option("display.max_colwidth", None)
records[["input", "output"] + feedback]

,input,output,Groundedness,Answer Relevance,Context Relevance
0,"""What are the keys to building a career in AI?""","""The keys to building a career in AI involve learning foundational technical skills, working on projects, finding a job, and being part of a supportive community.""",0.5,1.0,0.9
1,"""How can teamwork contribute to success in AI?""","""Teammates play a crucial role in the success of AI projects. Working collaboratively with colleagues who are dedicated, continuously learning, and focused on using AI to benefit everyone can positively influence one's own work ethic and outcomes. The ability to work effectively in a team, leveraging each member's strengths and insights, can lead to improved project outcomes and personal growth as a leader in AI projects.""",0.6,0.9,0.8
2,"""What is the importance of networking in AI?""","""Networking in AI is crucial as it can provide valuable insights, guidance, and opportunities for individuals looking to advance in the field. By connecting with professionals who have experience in AI, individuals can gain knowledge about the industry, potential career paths, and current trends. Networking also allows for the exchange of ideas, collaboration on projects, and access to job opportunities that may not be advertised publicly. Additionally, building a strong network in AI can help individuals stay updated on the latest developments and innovations in the field.""",0.0,1.0,0.8
3,"""What are some good habits to develop for a successful career?""","""Developing good habits in areas such as eating, exercise, sleep, personal relationships, work, learning, and self-care can help individuals move forward in their careers while maintaining their health.""",NaN,1.0,NaN
4,"""How can altruism be beneficial in building a career?""","""Helping others and lifting them up during one's own career journey can lead to better outcomes for oneself.""",NaN,1.0,NaN
5,"""What is imposter syndrome and how does it relate to AI?""","""Imposter syndrome is a phenomenon where individuals doubt their accomplishments and have a persistent fear of being exposed as a fraud, despite evidence of their success. In the context of AI, newcomers to the field may experience imposter syndrome, feeling like they do not truly belong in the AI community even if they have achieved success. This can be a common experience for individuals in AI, as highlighted by various successful figures who have openly discussed their own struggles with imposter syndrome.""",NaN,1.0,NaN
6,"""Who are some accomplished individuals who have experienced imposter syndrome?""","""Former Facebook COO Sheryl Sandberg, U.S. first lady Michelle Obama, actor Tom Hanks, and Atlassian co-CEO Mike Cannon-Brookes are some accomplished individuals who have experienced imposter syndrome.""",NaN,NaN,NaN
7,"""What is the first step to becoming good at AI?""","""Learning foundational technical skills.""",NaN,NaN,NaN
8,"""What are some common challenges in AI?""","""Some common challenges in AI include keeping up-to-date with rapidly evolving technologies, the iterative nature of AI projects leading to uncertainties in project management, difficulties in finding suitable projects and estimating timelines, collaborating with stakeholders who may lack expertise in AI, and the need for continuous learning throughout one's career.""",NaN,NaN,NaN
9,"""Is it normal to find parts of AI challenging?""","""It is normal to find parts of AI challenging.""",NaN,NaN,NaN


In [25]:
tru.get_leaderboard(app_ids=[])

,Groundedness,Answer Relevance,Context Relevance,latency,total_cost
app_id,,,,,
App_1,0.366667,0.983333,0.833333,1.545455,0.000841


In [26]:
tru.run_dashboard()

Starting dashboard ...
Config file already exists. Skipping writing process.
Credentials file already exists. Skipping writing process.


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at https://s172-29-58-181p38560.lab-aws-production.deeplearning.ai/ .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>